In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 · Calidad de datos
# MAGIC
# MAGIC Evidencia de cómo el pipeline `horoz_fraude_pipeline` detecta y maneja problemas de calidad:
# MAGIC
# MAGIC | Sección | Pregunta |
# MAGIC |---|---|
# MAGIC | 1 | ¿Cuántos registros hay en cada capa? |
# MAGIC | 2 | ¿Cuántos duplicados y llaves incompletas se eliminaron? |
# MAGIC | 3 | ¿Qué reportan las expectations en el event log del pipeline? |
# MAGIC | 4 | ¿Cuadran esas cifras al recalcularlas directo sobre las tablas? |
# MAGIC | 5 | ¿Qué columnas tienen valores nulos? |
# MAGIC | 6 | ¿Llegaron campos fuera del esquema (`_rescued_data`)? |
# MAGIC | 7 | ¿Qué formatos inconsistentes se normalizaron? |
# MAGIC | 8 | ¿Qué valores atípicos quedan? |
# MAGIC | 9 | Resumen para el reporte |
# MAGIC
# MAGIC **Requisito:** el pipeline debe haber terminado sin errores. Conectar a cómputo **Serverless**.

# COMMAND ----------

from pyspark.sql import functions as F

CATALOGO, ESQUEMA = "workspace", "horoz"
T = lambda nombre: f"{CATALOGO}.{ESQUEMA}.{nombre}"
KEY = ["LocationId", "BusinessDate", "Id"]

resumen = {}   # se llena en cada sección y se muestra al final

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Registros por capa

# COMMAND ----------

tablas = ["bronze_checks", "bronze_employees", "bronze_items", "bronze_locations", "bronze_revenue_centers",
          "silver_checks", "silver_items", "silver_voids", "silver_discounts",
          "gold_daily_sales", "gold_employee_activity", "gold_employee_weekly_risk", "gold_checks_to_review"]

conteos = []
for t in tablas:
    capa = t.split("_")[0]
    try:
        conteos.append((capa, t, spark.table(T(t)).count()))
    except Exception as e:
        conteos.append((capa, t, None))
        print(f"No se pudo leer {t}: {str(e)[:150]}")

display(spark.createDataFrame(conteos, "capa string, tabla string, registros long"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Duplicados y llaves incompletas
# MAGIC La llave de un ticket es `LocationId + BusinessDate + Id`. `checks_dedup` quita copias exactas
# MAGIC con `dropDuplicates` y la expectation `llave_completa` (DROP) elimina tickets sin llave.

# COMMAND ----------

bronze = spark.table(T("bronze_checks"))

n_bronze     = bronze.count()
n_llave_nula = bronze.filter("Id IS NULL OR LocationId IS NULL OR BusinessDate IS NULL").count()
n_distintos  = bronze.select(*KEY).distinct().count()
n_silver     = spark.table(T("silver_checks")).count()
n_duplicados = n_bronze - n_distintos

print(f"Tickets en bronze:                 {n_bronze:>12,}")
print(f"Copias duplicadas (misma llave):   {n_duplicados:>12,}")
print(f"Tickets con llave incompleta:      {n_llave_nula:>12,}")
print(f"Tickets en silver_checks:          {n_silver:>12,}")
print(f"Eliminados en total:               {n_bronze - n_silver:>12,}")

resumen["duplicados"] = n_duplicados
resumen["llave_nula"] = n_llave_nula

# COMMAND ----------

# Ejemplos de duplicados: ¿vienen del mismo archivo o de archivos distintos?
display(bronze.groupBy(*KEY)
        .agg(F.count("*").alias("copias"),
             F.countDistinct("source_file").alias("archivos_distintos"),
             F.collect_set("source_file").alias("archivos"))
        .filter("copias > 1")
        .orderBy(F.desc("copias"))
        .limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Expectations según el event log del pipeline
# MAGIC Toma la **última actualización exitosa** del pipeline. Si la última corrida fue un full refresh,
# MAGIC las cifras cubren todos los datos.

# COMMAND ----------

EVENT_LOG = f"event_log(TABLE({T('silver_checks')}))"

try:
    historial = spark.sql(f"""
        SELECT origin.update_id,
               MIN(timestamp) AS inicio,
               MAX(timestamp) AS fin,
               MAX_BY(details:update_progress:state, timestamp) AS estado_final
        FROM {EVENT_LOG}
        WHERE event_type = 'update_progress'
        GROUP BY origin.update_id
        ORDER BY inicio DESC
    """)
    print("Historial de actualizaciones del pipeline (incluye las fallidas):")
    display(historial.limit(10))

    ultima_ok = (historial.filter("estado_final = 'COMPLETED'")
                 .orderBy(F.desc("inicio")).first())
    if ultima_ok is None:
        print("No hay ninguna actualización COMPLETED todavía.")
    else:
        print(f"Usando la actualización {ultima_ok.update_id} ({ultima_ok.inicio})")
        expectations_log = spark.sql(f"""
            SELECT e.dataset, e.name AS expectation,
                   SUM(e.passed_records) AS pasaron,
                   SUM(e.failed_records) AS fallaron,
                   ROUND(100 * SUM(e.failed_records) / (SUM(e.passed_records) + SUM(e.failed_records)), 3) AS pct_falla
            FROM (
              SELECT explode(from_json(details:flow_progress:data_quality:expectations,
                     'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>')) AS e
              FROM {EVENT_LOG}
              WHERE event_type = 'flow_progress'
                AND origin.update_id = '{ultima_ok.update_id}'
            )
            GROUP BY e.dataset, e.name
            ORDER BY fallaron DESC
        """)
        display(expectations_log)
except Exception as e:
    print("No se pudo leer el event log. Debe correrlo el dueño del pipeline o alguien con permisos sobre él.")
    print(str(e)[:300])

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Recalcular las reglas directo sobre las tablas
# MAGIC Comprueba las cifras del event log de forma independiente. Se separan las filas donde la regla
# MAGIC **falla** de las filas donde no se puede evaluar porque algún campo es nulo.

# COMMAND ----------

reglas_checks = {
    "horas_validas":               "close_ts >= open_ts",
    "pago_cuadra":                 "abs(dif_pago) <= 0.05",
    "comensales_en_comedor":       "RevenueCenterId <> 1 OR TrafficCount > 0",
    "un_desc_empleado_por_ticket": "n_desc_empleado <= 1",
    "desc_empleado_no_combinado":  "n_desc_empleado = 0 OR n_desc_otros = 0",
    "void_no_combinado_con_desc":  "n_voids = 0 OR n_desc_total = 0",
}
reglas_items = {"precio_no_negativo": "SoldPrice >= 0"}


def evaluar(tabla, reglas):
    df = spark.table(T(tabla))
    aggs = [F.count("*").alias("_total")]
    for nombre, cond in reglas.items():
        aggs += [F.sum(F.when(F.expr(cond) == F.lit(False), 1).otherwise(0)).alias(f"{nombre}__falla"),
                 F.sum(F.when(F.expr(cond).isNull(), 1).otherwise(0)).alias(f"{nombre}__nulo")]
    r = df.agg(*aggs).first().asDict()
    total = r["_total"]
    return [(tabla, n, c, r[f"{n}__falla"], r[f"{n}__nulo"], round(100 * r[f"{n}__falla"] / total, 3))
            for n, c in reglas.items()]


filas = evaluar("silver_checks", reglas_checks) + evaluar("silver_items", reglas_items)
reglas_df = spark.createDataFrame(
    filas, "tabla string, expectation string, condicion string, fallan long, no_evaluables_por_nulo long, pct_falla double")
display(reglas_df.orderBy(F.desc("fallan")))

for f in filas:
    resumen[f[1]] = f[3]

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Valores nulos por columna
# MAGIC Solo se muestran columnas con al menos un nulo. En `silver_checks` los montos se calculan con
# MAGIC `coalesce(..., 0)`, así que un nulo en el JSON original se convierte en 0 y no rompe las sumas.

# COMMAND ----------

def nulos(tabla):
    df = spark.table(T(tabla))
    cols = [c for c, t in df.dtypes if not t.startswith(("array", "struct", "map"))]
    fila = df.select([F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in cols]
                     + [F.count("*").alias("__total")]).first().asDict()
    total = fila.pop("__total")
    return [(tabla, c, v, round(100 * v / total, 2)) for c, v in fila.items() if v]


filas_nulos = []
for t in ["silver_checks", "silver_items", "silver_voids", "silver_discounts"]:
    filas_nulos += nulos(t)

nulos_df = spark.createDataFrame(filas_nulos, "tabla string, columna string, nulos long, pct double")
display(nulos_df.orderBy("tabla", F.desc("nulos")))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Campos fuera del esquema (`_rescued_data`)
# MAGIC Auto Loader guarda aquí cualquier campo nuevo o con tipo distinto al esperado, en vez de perderlo.

# COMMAND ----------

if "_rescued_data" in bronze.columns:
    rescatados = bronze.filter("_rescued_data IS NOT NULL")
    n_rescatados = rescatados.count()
    print(f"Tickets con datos rescatados: {n_rescatados:,}")
    resumen["rescued_data"] = n_rescatados
    if n_rescatados:
        display(rescatados.select(*KEY, "_rescued_data", "source_file").limit(20))
else:
    print("bronze_checks no tiene columna _rescued_data.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Formatos inconsistentes normalizados

# COMMAND ----------

# MAGIC %md
# MAGIC ### 7a. Categorías de void: valor crudo → valor normalizado

# COMMAND ----------

voids_crudos = (bronze.select(F.explode("Voids").alias("v"))
                .select(F.col("v.VoidCategoryName").alias("crudo"))
                .withColumn("normalizado", F.upper(F.trim("crudo")))
                .withColumn("normalizado",
                            F.when(F.col("normalizado") == "SERV ERR", "SERVER ERR")
                             .when(F.col("normalizado") == "NOT RECEVD", "NOT RECVD")
                             .otherwise(F.col("normalizado"))))

print("Categorías distintas antes:", voids_crudos.select("crudo").distinct().count())
print("Categorías distintas después:", voids_crudos.select("normalizado").distinct().count())
display(voids_crudos.groupBy("crudo", "normalizado").count()
        .withColumn("cambio", F.col("crudo") != F.col("normalizado"))
        .orderBy("normalizado", F.desc("count")))

# COMMAND ----------

# MAGIC %md
# MAGIC ### 7b. Nombres de descuento y su tipo
# MAGIC `promocion` es la categoría por defecto: revisar que no esconda descuentos de empleado o gerente mal escritos.

# COMMAND ----------

desc = spark.table(T("silver_discounts"))
crudos = bronze.select(F.explode("Discounts").alias("d")).select(F.col("d.DiscountName").alias("n"))
print("Nombres distintos antes:", crudos.distinct().count())
print("Nombres distintos después (upper + trim):", desc.select("DiscountName").distinct().count())

display(desc.groupBy("tipo_descuento")
        .agg(F.count("*").alias("descuentos"),
             F.round(F.sum("Amount"), 2).alias("monto"),
             F.countDistinct("DiscountName").alias("nombres_distintos"))
        .orderBy(F.desc("descuentos")))

# COMMAND ----------

display(desc.filter("tipo_descuento = 'promocion'")
        .groupBy("DiscountName").agg(F.count("*").alias("descuentos"), F.round(F.sum("Amount"), 2).alias("monto"))
        .orderBy(F.desc("descuentos")).limit(30))

# COMMAND ----------

# MAGIC %md
# MAGIC ### 7c. Fechas y horas que no se pudieron convertir

# COMMAND ----------

chk = spark.table(T("silver_checks"))
display(chk.agg(
    F.sum((F.col("OpenTime").isNotNull() & F.col("open_ts").isNull()).cast("int")).alias("open_no_convertible"),
    F.sum((F.col("CloseTime").isNotNull() & F.col("close_ts").isNull()).cast("int")).alias("close_no_convertible"),
    F.sum(F.col("BusinessDate").isNull().cast("int")).alias("business_date_nula")))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Valores atípicos que se conservan

# COMMAND ----------

display(chk.agg(
    F.sum((F.col("venta") <= 0).cast("int")).alias("tickets_venta_cero_o_negativa"),
    F.sum((F.col("min_mesa") < 0).cast("int")).alias("min_mesa_negativo"),
    F.sum((F.col("min_mesa") > 480).cast("int")).alias("min_mesa_mas_de_8h"),
    F.sum(((F.col("RevenueCenterId") == 1) & (F.col("TrafficCount") > 20)).cast("int")).alias("comedor_mas_de_20_comensales"),
    F.expr("percentile_approx(min_mesa, array(0.5, 0.95, 0.99))").alias("min_mesa_p50_p95_p99")))

# COMMAND ----------

# Productos con precio negativo: ¿qué son?
display(spark.table(T("silver_items")).filter("SoldPrice < 0")
        .groupBy("ItemName").agg(F.count("*").alias("veces"), F.round(F.sum("SoldPrice"), 2).alias("monto"))
        .orderBy(F.desc("veces")).limit(20))

# COMMAND ----------

# Tickets de comedor con 0 comensales: ¿problema de una sucursal o de todas?
loc = spark.table(T("bronze_locations")).select(F.col("Id").alias("LocationId"), F.col("Name").alias("sucursal"))
display(chk.filter("RevenueCenterId = 1")
        .groupBy("LocationId")
        .agg(F.count("*").alias("tickets_comedor"),
             F.sum((F.coalesce("TrafficCount", F.lit(0)) <= 0).cast("int")).alias("sin_comensales"))
        .withColumn("pct_sin_comensales", F.round(100 * F.col("sin_comensales") / F.col("tickets_comedor"), 1))
        .join(loc, "LocationId", "left")
        .orderBy(F.desc("pct_sin_comensales")).limit(20))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Resumen para el reporte

# COMMAND ----------

def g(k):
    return resumen.get(k)

filas_resumen = [
    ("Duplicados exactos",                   g("duplicados"),                  "Eliminados con dropDuplicates por llave (checks_dedup)"),
    ("Llave incompleta",                     g("llave_nula"),                  "expect_or_drop llave_completa"),
    ("Comedor sin comensales",               g("comensales_en_comedor"),       "Expectation ALLOW; gold solo suma comensales de comedor"),
    ("Void + descuento en el mismo ticket",  g("void_no_combinado_con_desc"),  "Expectation ALLOW; cuenta como violación en el risk score"),
    ("Descuento de empleado combinado",      g("desc_empleado_no_combinado"),  "Expectation ALLOW; cuenta como violación en el risk score"),
    ("Más de un descuento de empleado",      g("un_desc_empleado_por_ticket"), "Expectation ALLOW; cuenta como violación en el risk score"),
    ("Pago no cuadra",                       g("pago_cuadra"),                 "Expectation ALLOW; aislados en gold_checks_to_review"),
    ("Hora de cierre antes de apertura",     g("horas_validas"),               "Expectation ALLOW"),
    ("Precio de producto negativo",          g("precio_no_negativo"),          "Expectation ALLOW en silver_items"),
    ("Campos fuera de esquema",              g("rescued_data"),                "Capturados en _rescued_data por Auto Loader"),
]
display(spark.createDataFrame(filas_resumen, "problema string, registros long, manejo string"))